In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/deliveries.csv")

print(df.shape)
print(df["label"].value_counts(normalize=True))
print(df.describe())

(958143, 11)
label
0    0.511784
1    0.488216
Name: proportion, dtype: float64
           match_id           ball  balls_remaining    runs_needed  \
count  9.581430e+05  958143.000000    958143.000000  958143.000000   
mean   9.574165e+05      16.660079       132.750781     128.731472   
std    4.789546e+05      12.521201        83.245724      78.072928   
min    6.481400e+04       0.100000         0.000000     -83.000000   
25%    4.766000e+05       6.500000        65.000000      68.000000   
50%    1.144527e+06      13.600000       111.000000     119.000000   
75%    1.383077e+06      25.100000       203.000000     180.000000   
max    1.542990e+06      49.800000       300.000000     499.000000   

       wickets_in_hand  current_run_rate  required_run_rate          label  
count    958143.000000     958143.000000      958143.000000  958143.000000  
mean          7.102083          5.661173           7.167630       0.488216  
std           2.365952          2.363656           5.16724

In [2]:
df.sort_values("required_run_rate", ascending=False).head(10)

,match_id,batting_team,bowling_team,match_type,ball,balls_remaining,runs_needed,wickets_in_hand,current_run_rate,required_run_rate,label
283758,247490,South Africa,Bangladesh,ODI,48.1,12,73,1,3.729167,36.0,0
283759,247490,South Africa,Bangladesh,ODI,48.2,11,73,1,3.716263,36.0,0
283760,247490,South Africa,Bangladesh,ODI,48.3,11,68,1,3.820069,36.0,0
283761,247490,South Africa,Bangladesh,ODI,48.4,10,68,1,3.806897,36.0,0
283762,247490,South Africa,Bangladesh,ODI,48.5,9,68,0,3.793814,36.0,0
958014,995467,Sri Lanka,Australia,T20,16.6,18,111,3,9.000000,36.0,0
958015,995467,Sri Lanka,Australia,T20,17.1,17,107,3,9.145631,36.0,0
284570,247493,West Indies,South Africa,ODI,47.4,14,84,1,5.727273,36.0,0
284571,247493,West Indies,South Africa,ODI,47.5,13,84,1,5.707317,36.0,0
284572,247493,West Indies,South Africa,ODI,47.6,12,82,1,5.729167,36.0,0


In [3]:
match_ids = df["match_id"].unique()
print(len(match_ids))

5720


In [4]:
from sklearn.model_selection import train_test_split

train_match_ids, test_match_ids = train_test_split(match_ids, test_size=0.2)
print(len(train_match_ids), len(test_match_ids))

4576 1144


In [5]:
train_df = df[df["match_id"].isin(train_match_ids)]
test_df = df[df["match_id"].isin(test_match_ids)]

print(len(train_df), len(test_df))
print(len(train_df) + len(test_df), len(df))

764291 193852
958143 958143


In [6]:
train_df["match_type"] = train_df["match_type"].map({"ODI": 0, "T20": 1})
test_df["match_type"] = test_df["match_type"].map({"ODI": 0, "T20": 1})

In [7]:
feature_columns = ["match_type", "balls_remaining", "runs_needed", "wickets_in_hand", "current_run_rate", "required_run_rate"]

X_train = train_df[feature_columns]
y_train = train_df["label"]

X_test = test_df[feature_columns]
y_test = test_df["label"]

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(764291, 6) (764291,)
(193852, 6) (193852,)


In [8]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default sol

In [9]:
y_pred = model.predict(X_test)

from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.8163


In [10]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.83      0.82      0.82    102111
           1       0.80      0.81      0.81     91741

    accuracy                           0.82    193852
   macro avg       0.82      0.82      0.82    193852
weighted avg       0.82      0.82      0.82    193852



In [11]:
print(X_test.isna().sum())
print(y_test.isna().sum())

match_type           0
balls_remaining      0
runs_needed          0
wickets_in_hand      0
current_run_rate     0
required_run_rate    0
dtype: int64
0


In [12]:
print(len(X_test), len(y_test), len(y_pred))


193852 193852 193852


In [13]:
train_match_ids, test_match_ids = train_test_split(match_ids, test_size=0.2, random_state=42)

In [15]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstr

In [16]:
rf_pred = rf_model.predict(X_test)

print(classification_report(y_test, rf_pred))

              precision    recall  f1-score   support

           0       0.81      0.80      0.80    102111
           1       0.78      0.78      0.78     91741

    accuracy                           0.79    193852
   macro avg       0.79      0.79      0.79    193852
weighted avg       0.79      0.79      0.79    193852



In [17]:
rf_train_pred = rf_model.predict(X_train)
print("Train accuracy:", accuracy_score(y_train, rf_train_pred))
print("Test accuracy:", accuracy_score(y_test, rf_pred))

Train accuracy: 0.9845373031999591
Test accuracy: 0.7922590429812434


In [18]:
rf_model_v2 = RandomForestClassifier(random_state=42, max_depth=10, min_samples_leaf=50)
rf_model_v2.fit(X_train, y_train)

rf_v2_train_pred = rf_model_v2.predict(X_train)
rf_v2_test_pred = rf_model_v2.predict(X_test)

print("Train accuracy:", accuracy_score(y_train, rf_v2_train_pred))
print("Test accuracy:", accuracy_score(y_test, rf_v2_test_pred))

Train accuracy: 0.8374768249266313
Test accuracy: 0.8228287559581536


In [19]:
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=5)

for train_idx, val_idx in gkf.split(X_train, y_train, groups=train_df["match_id"]):
    print(len(train_idx), len(val_idx))

611431 152860
611435 152856
611436 152855
611431 152860
611431 152860


In [20]:
param_options = [
    {"max_depth": 5, "min_samples_leaf": 50},
    {"max_depth": 10, "min_samples_leaf": 50},
    {"max_depth": 15, "min_samples_leaf": 50},
]

for params in param_options:
    fold_accuracies = []
    for train_idx, val_idx in gkf.split(X_train, y_train, groups=train_df["match_id"]):
        X_fold_train = X_train.iloc[train_idx]
        y_fold_train = y_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]
        y_fold_val = y_train.iloc[val_idx]

        model = RandomForestClassifier(random_state=42, **params)
        model.fit(X_fold_train, y_fold_train)
        fold_pred = model.predict(X_fold_val)
        fold_accuracies.append(accuracy_score(y_fold_val, fold_pred))

    print(params, "avg accuracy:", sum(fold_accuracies) / len(fold_accuracies))

{'max_depth': 5, 'min_samples_leaf': 50} avg accuracy: 0.8076662489426901
{'max_depth': 10, 'min_samples_leaf': 50} avg accuracy: 0.8227468845945513
{'max_depth': 15, 'min_samples_leaf': 50} avg accuracy: 0.8196054095360218


In [21]:
final_model = RandomForestClassifier(random_state=42, max_depth=10, min_samples_leaf=50)
final_model.fit(X_train, y_train)

final_pred = final_model.predict(X_test)
print(classification_report(y_test, final_pred))

              precision    recall  f1-score   support

           0       0.83      0.84      0.83    102111
           1       0.82      0.81      0.81     91741

    accuracy                           0.82    193852
   macro avg       0.82      0.82      0.82    193852
weighted avg       0.82      0.82      0.82    193852

